# Plot Hourly FE (No Mutation)

This notebook reads the hourly FE CSV files written by the new no-mutation simulation notebook.

Use it in this order:

1. update the **Parameter Setting** cell
2. run the **List available scenarios** cell
3. set `scenario_name`
4. load the FE file
5. plot the seller and/or buyer hourly FE distribution


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde


In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


## Parameter Setting

Update only this cell first.


In [ ]:
# ============================================================
# Required inputs
# ============================================================
# Folder written by the new no-mutation simulation notebook.
output_dir = "Output files (Risk Neutral, No Mutation, Verified)"

# Seller-buyer match ID.
match_id = 1

# Exact scenario name from Hourly_FE_Index.csv.
# Leave as None first, run the scenario-list cell, then paste one exact name here.
scenario_name: Optional[str] = "Baseline__No_Mutation__Verified"

# ============================================================
# Risk parameters used only for plotting the risk-adjusted FE line
# ============================================================
lambda_seller = 0.0
lambda_buyer = 0.0

# ============================================================
# Plot controls
# ============================================================
# Options: "both", "seller", "buyer"
plot_party = "both"

# Optional clipping only for the displayed curve.
clip_for_plot = True
lower_q = 0.005
upper_q = 0.995

# Add timestamps back into the loaded FE table.
add_timestamp = False

# Timestamp reconstruction mode:
#   "original_match" -> use the timestamp column from Match files of original data
#   "simulation_start" -> reconstruct from simulation_start
timestamp_mode = "original_match"

simulation_start = "2025-01-01 00:00:00"
original_match_folder_name = "Match files of original data"

# Save figures
save_fig = False
save_fig_dir_name = "Figures_Hourly_FE"

# Histogram / density settings
plot_histogram = True
plot_density = True
hist_bins = 80

SETTINGS_FE = {
    "output_dir": output_dir,
    "match_id": int(match_id),
    "scenario_name": scenario_name,
    "lambda_seller": float(lambda_seller),
    "lambda_buyer": float(lambda_buyer),
    "plot_party": plot_party,
    "clip_for_plot": bool(clip_for_plot),
    "lower_q": float(lower_q),
    "upper_q": float(upper_q),
    "add_timestamp": bool(add_timestamp),
    "timestamp_mode": timestamp_mode,
    "simulation_start": simulation_start,
    "original_match_folder_name": original_match_folder_name,
    "save_fig": bool(save_fig),
    "save_fig_dir_name": save_fig_dir_name,
    "plot_histogram": bool(plot_histogram),
    "plot_density": bool(plot_density),
    "hist_bins": int(hist_bins),
}
SETTINGS_FE


## Helper functions


In [ ]:
NOTEBOOK_CWD = Path.cwd().resolve()
BUNDLE_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "simulation_baseline" else NOTEBOOK_CWD


def _search_roots(max_parent_depth: int = 4) -> list[Path]:
    roots = []
    seen = set()
    for anchor in [BUNDLE_ROOT, NOTEBOOK_CWD, Path("/mnt/data")]:
        p = Path(anchor).expanduser()
        for root in [p, *list(p.parents)[:max_parent_depth]]:
            key = str(root)
            if key not in seen:
                seen.add(key)
                roots.append(root)
    return roots


def _integer_stat_label(value: float) -> str:
    return f"{int(np.rint(float(value))):,}"


def _coerce_numeric_or_nan(value) -> float:
    if value is None:
        return np.nan
    if isinstance(value, str) and value.strip().upper() in {"", "N/A", "NA", "NONE"}:
        return np.nan
    try:
        return float(value)
    except Exception:
        return np.nan


def _output_dir_path(output_dir: str | Path) -> Path:
    path = Path(output_dir).expanduser()
    if path.exists():
        return path.resolve()
    for root in _search_roots():
        alt = root / path
        if alt.exists():
            return alt.resolve()
    raise FileNotFoundError(f"Output directory was not found: {output_dir}")


def _hourly_fe_index_file(output_dir: str | Path) -> Path:
    out_dir = _output_dir_path(output_dir)
    index_file = out_dir / "Hourly_FE_Index.csv"
    if not index_file.exists():
        raise FileNotFoundError(f"Hourly FE index file not found: {index_file}")
    return index_file


def _resolve_original_match_dir(folder_name: str) -> Optional[Path]:
    candidates = []
    for root in _search_roots():
        candidates.extend([
            root / folder_name,
            root / "Input data and files" / folder_name,
            root / "Data and files" / folder_name,
        ])
    for candidate in candidates:
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    return None


def list_saved_hourly_fe_scenarios(match_id: int, output_dir: str | Path) -> pd.DataFrame:
    index_df = read_csv_optimized(_hourly_fe_index_file(output_dir))
    index_df["match_id"] = pd.to_numeric(index_df["match_id"], errors="coerce").astype("Int64")
    out = index_df.loc[index_df["match_id"] == int(match_id)].copy()
    return out.reset_index(drop=True)


def get_saved_hourly_fe_index_row(match_id: int, scenario_name: str, output_dir: str | Path) -> pd.Series:
    fe_index_df = list_saved_hourly_fe_scenarios(match_id, output_dir)
    scenario_mask = fe_index_df["scenario_name"].astype(str) == str(scenario_name)
    matched_rows = fe_index_df.loc[scenario_mask].copy()

    if matched_rows.empty:
        available = fe_index_df["scenario_name"].astype(str).tolist()
        raise KeyError(
            f"Scenario {scenario_name!r} was not found for match {int(match_id):04d}. "
            f"Available scenarios: {available}"
        )

    return matched_rows.iloc[0].copy()


def _reconstruct_timestamps_from_original_match(match_id: int,
                                                hours_per_replication: int,
                                                replication: pd.Series,
                                                hour_index: pd.Series,
                                                folder_name: str) -> Optional[pd.Series]:
    match_dir = _resolve_original_match_dir(folder_name)
    if match_dir is None:
        return None

    candidate = match_dir / f"{int(match_id):03d}.csv"
    if not candidate.exists():
        alternatives = list(match_dir.glob(f"*{int(match_id):03d}*.csv"))
        if alternatives:
            candidate = alternatives[0]
    if not candidate.exists():
        return None

    original_df = read_csv_optimized(candidate)
    if "timestamp" not in original_df.columns:
        return None

    timestamps = pd.to_datetime(original_df["timestamp"], errors="coerce")
    if timestamps.isna().all():
        return None

    timestamps = timestamps.iloc[:hours_per_replication].reset_index(drop=True)
    if len(timestamps) != int(hours_per_replication):
        return None

    rebuilt = []
    for rep, h in zip(replication.astype(int), hour_index.astype(int)):
        idx = int(h) - 1
        if idx < 0 or idx >= len(timestamps):
            rebuilt.append(pd.NaT)
        else:
            rebuilt.append(timestamps.iloc[idx])
    return pd.Series(rebuilt)


def load_saved_hourly_fe(match_id: int,
                         scenario_name: str,
                         output_dir: str | Path,
                         add_timestamp: bool = False,
                         timestamp_mode: str = "original_match",
                         simulation_start: Optional[str] = None,
                         original_match_folder_name: str = "Match files of original data") -> Tuple[pd.DataFrame, pd.Series]:
    index_row = get_saved_hourly_fe_index_row(match_id, scenario_name, output_dir)

    fe_file = Path(index_row["hourly_fe_file"]).expanduser()
    if not fe_file.exists():
        out_dir = _output_dir_path(output_dir)
        fe_file = out_dir / "Hourly_FE" / fe_file.name
    if not fe_file.exists():
        raise FileNotFoundError(f"Saved hourly FE file not found: {fe_file}")

    hourly_fe_df = read_csv_optimized(fe_file)
    hourly_fe_df["replication"] = pd.to_numeric(hourly_fe_df["replication"], errors="coerce").astype("Int64")
    hourly_fe_df["hour_index"] = pd.to_numeric(hourly_fe_df["hour_index"], errors="coerce").astype("Int64")

    if add_timestamp and "timestamp" not in hourly_fe_df.columns:
        hours_per_rep = int(index_row.get("hours_per_replication", 0))
        mode = str(timestamp_mode).strip().lower()

        if mode == "original_match":
            rebuilt = _reconstruct_timestamps_from_original_match(
                match_id=match_id,
                hours_per_replication=hours_per_rep,
                replication=hourly_fe_df["replication"],
                hour_index=hourly_fe_df["hour_index"],
                folder_name=original_match_folder_name,
            )
            if rebuilt is not None:
                hourly_fe_df["timestamp"] = rebuilt
            else:
                mode = "simulation_start"

        if mode == "simulation_start":
            start_timestamp = pd.Timestamp(simulation_start if simulation_start is not None else SETTINGS_FE["simulation_start"])
            full_hour_index = pd.to_numeric(hourly_fe_df["hour_index"], errors="coerce") - 1
            hourly_fe_df["timestamp"] = start_timestamp + pd.to_timedelta(full_hour_index, unit="h")

    return hourly_fe_df, index_row


def _build_ppa_label(index_row: pd.Series) -> str:
    ppa_type = str(index_row.get("ppa_type", "Unknown"))
    profile_type = str(index_row.get("profile_type", "Unknown"))
    strike_price = _coerce_numeric_or_nan(index_row.get("strike_price_mwh"))
    fixed_volume = _coerce_numeric_or_nan(index_row.get("volume_mw"))

    parts = [f"{ppa_type}-{profile_type}"]

    if pd.notna(strike_price):
        parts.append(f"Strike = {_integer_stat_label(strike_price)}")

    if str(profile_type).strip().lower() == "fix" and pd.notna(fixed_volume):
        parts.append(f"Volume = {_integer_stat_label(fixed_volume)}")

    return " | ".join(parts)


def _build_plot_series(series: pd.Series,
                       clip_for_plot: bool = True,
                       lower_q: float = 0.005,
                       upper_q: float = 0.995) -> Tuple[pd.Series, float, float]:
    series = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if series.empty:
        return series, np.nan, np.nan

    if (not clip_for_plot) or (series.nunique() <= 1):
        return series, float(series.min()), float(series.max())

    x_low = float(series.quantile(lower_q))
    x_high = float(series.quantile(upper_q))
    if not np.isfinite(x_low) or not np.isfinite(x_high) or x_high <= x_low:
        return series, float(series.min()), float(series.max())

    plot_series = series[(series >= x_low) & (series <= x_high)].copy()
    if plot_series.empty:
        return series, float(series.min()), float(series.max())

    return plot_series, x_low, x_high


def _print_plot_info(match_id: int, scenario_name: str, index_row: pd.Series,
                     party_label: str, ppa_label: str,
                     mean_val: float, std_val: float, risk_line: float) -> None:
    print(f"Match ID: {int(match_id):04d}")
    print(f"Party: {party_label}")
    print(f"PPA: {ppa_label}")
    print(f"Scenario: {scenario_name}")
    print(f"FE source: {index_row.get('fe_source', '')}")
    print(f"Mean: {_integer_stat_label(mean_val)}")
    print(f"Std.: {_integer_stat_label(std_val)}")
    print(f"Risk-adjusted FE: {_integer_stat_label(risk_line)}")
    print("-" * 80)


def plot_hourly_fe_distribution_from_saved_data(hourly_fe_df: pd.DataFrame,
                                                index_row: pd.Series,
                                                match_id: int,
                                                scenario_name: str,
                                                party_label: str,
                                                lambda_value: float,
                                                output_dir: str | Path,
                                                clip_for_plot: bool = True,
                                                lower_q: float = 0.005,
                                                upper_q: float = 0.995,
                                                plot_histogram: bool = True,
                                                plot_density: bool = True,
                                                hist_bins: int = 80,
                                                save_fig: bool = False,
                                                save_fig_dir_name: str = "Figures_Hourly_FE") -> None:
    column_name = "seller_fe" if str(party_label).strip().lower() == "seller" else "buyer_fe"
    series = pd.to_numeric(hourly_fe_df[column_name], errors="coerce").dropna()

    if series.empty:
        print(f"Skipping {scenario_name} / {party_label}: no valid FE values.")
        return

    mean_val = float(series.mean())
    std_val = float(series.std(ddof=0))
    risk_line = mean_val + float(lambda_value) * std_val

    plot_series, x_low, x_high = _build_plot_series(
        series=series,
        clip_for_plot=clip_for_plot,
        lower_q=lower_q,
        upper_q=upper_q,
    )

    if pd.isna(x_low) or pd.isna(x_high):
        x_low = float(series.min())
        x_high = float(series.max())

    x_low = min(x_low, mean_val, risk_line, 0.0)
    x_high = max(x_high, mean_val, risk_line, 0.0)
    x_pad = 0.05 * (x_high - x_low) if x_high > x_low else 1.0
    x_low -= x_pad
    x_high += x_pad

    plt.figure(figsize=(10, 6))

    if plot_histogram:
        plt.hist(plot_series, bins=int(hist_bins), density=True, alpha=0.35, label="Histogram")

    if plot_density and plot_series.nunique() > 1:
        kde = gaussian_kde(plot_series)
        x_grid = np.linspace(x_low, x_high, 600)
        y_grid = kde(x_grid)
        plt.plot(x_grid, y_grid, linewidth=2, label="Density curve")

    plt.axvline(mean_val, linestyle="--", linewidth=2, label=f"Mean = {_integer_stat_label(mean_val)}")
    plt.axvline(risk_line, linestyle="-.", linewidth=2, label=f"Risk-adjusted FE = {_integer_stat_label(risk_line)}")
    plt.axvline(0.0, linestyle="-", linewidth=2, label="_nolegend_")

    plt.xlim(x_low, x_high)
    plt.xlabel(f"{party_label} hourly FE")
    plt.ylabel("Probability density")
    plt.legend()
    plt.tight_layout()

    if save_fig:
        out_dir = _output_dir_path(output_dir) / save_fig_dir_name
        out_dir.mkdir(parents=True, exist_ok=True)
        fig_path = out_dir / f"Hourly_FE_Match_{int(match_id):04d}__{scenario_name}__{party_label}.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure: {fig_path}")

    plt.show()

    _print_plot_info(
        match_id=match_id,
        scenario_name=scenario_name,
        index_row=index_row,
        party_label=party_label,
        ppa_label=_build_ppa_label(index_row),
        mean_val=mean_val,
        std_val=std_val,
        risk_line=risk_line,
    )


## List available scenarios for this match


In [ ]:
fe_index_df = list_saved_hourly_fe_scenarios(SETTINGS_FE["match_id"], SETTINGS_FE["output_dir"])
fe_index_df


## Load one scenario

Set `scenario_name` in the Parameter Setting cell, then run this cell.


In [ ]:
if SETTINGS_FE["scenario_name"] is None:
    raise ValueError("Set scenario_name in the Parameter Setting cell, then run again.")

hourly_fe_df, index_row = load_saved_hourly_fe(
    match_id=SETTINGS_FE["match_id"],
    scenario_name=SETTINGS_FE["scenario_name"],
    output_dir=SETTINGS_FE["output_dir"],
    add_timestamp=SETTINGS_FE["add_timestamp"],
    timestamp_mode=SETTINGS_FE["timestamp_mode"],
    simulation_start=SETTINGS_FE["simulation_start"],
    original_match_folder_name=SETTINGS_FE["original_match_folder_name"],
)

print("Loaded scenario:")
print(f"  Match ID   : {SETTINGS_FE['match_id']:04d}")
print(f"  Scenario   : {SETTINGS_FE['scenario_name']}")
print(f"  FE source  : {index_row.get('fe_source', '')}")
print(f"  FE file    : {index_row.get('hourly_fe_file', '')}")
print(f"  Rows loaded: {len(hourly_fe_df):,}")
print(f"  PPA        : {_build_ppa_label(index_row)}")

index_row.to_frame(name="Value")


## Preview FE data


In [ ]:
hourly_fe_df.head()


## Plot FE distribution


In [ ]:
plot_mode = str(SETTINGS_FE["plot_party"]).strip().lower()
if plot_mode not in {"both", "seller", "buyer"}:
    raise ValueError("plot_party must be 'both', 'seller', or 'buyer'.")

if plot_mode in {"both", "seller"}:
    plot_hourly_fe_distribution_from_saved_data(
        hourly_fe_df=hourly_fe_df,
        index_row=index_row,
        match_id=SETTINGS_FE["match_id"],
        scenario_name=SETTINGS_FE["scenario_name"],
        party_label="Seller",
        lambda_value=SETTINGS_FE["lambda_seller"],
        output_dir=SETTINGS_FE["output_dir"],
        clip_for_plot=SETTINGS_FE["clip_for_plot"],
        lower_q=SETTINGS_FE["lower_q"],
        upper_q=SETTINGS_FE["upper_q"],
        plot_histogram=SETTINGS_FE["plot_histogram"],
        plot_density=SETTINGS_FE["plot_density"],
        hist_bins=SETTINGS_FE["hist_bins"],
        save_fig=SETTINGS_FE["save_fig"],
        save_fig_dir_name=SETTINGS_FE["save_fig_dir_name"],
    )

if plot_mode in {"both", "buyer"}:
    plot_hourly_fe_distribution_from_saved_data(
        hourly_fe_df=hourly_fe_df,
        index_row=index_row,
        match_id=SETTINGS_FE["match_id"],
        scenario_name=SETTINGS_FE["scenario_name"],
        party_label="Buyer",
        lambda_value=SETTINGS_FE["lambda_buyer"],
        output_dir=SETTINGS_FE["output_dir"],
        clip_for_plot=SETTINGS_FE["clip_for_plot"],
        lower_q=SETTINGS_FE["lower_q"],
        upper_q=SETTINGS_FE["upper_q"],
        plot_histogram=SETTINGS_FE["plot_histogram"],
        plot_density=SETTINGS_FE["plot_density"],
        hist_bins=SETTINGS_FE["hist_bins"],
        save_fig=SETTINGS_FE["save_fig"],
        save_fig_dir_name=SETTINGS_FE["save_fig_dir_name"],
    )
